# 11. 팀원 10_deflation 결과 + submit_15 순위 평균 앙상블
- 팀원 파일(s2/s3/s4/s5)은 `10_deflation.ipynb`를 로컬 실행해 생성 (팀원이 07 제출 csv 비교 출력만 생략한 사본으로 실행).
- submit_15는 리더보드에 낸 커밋 버전(LB 0.6110) 그대로 사용.
- 리더보드: submit_15 0.6110, submit_17 0.5990, 10_s2 0.6150, 0.6290(s3 또는 s4 — 팀 확인 필요)

In [1]:
import pandas as pd, numpy as np
from scipy.stats import rankdata
SUB = '../submissions'
files = {'s15': 'submit_15_M4fix_shockadj_previnv.csv', 's2': 'submit_10_s2_cons_defl.csv',
         's3': 'submit_10_s3_cons_defl_adj.csv', 's4': 'submit_10_s4_07_defl.csv', 's5': 'submit_10_s5_07_defl_adj.csv'}
sample = pd.read_csv('../data/raw/epoch_data/sample_submission.csv', encoding='utf-8-sig')
P = pd.DataFrame({k: pd.read_csv(f'{SUB}/{f}').set_index('row_id').prediction.loc[sample.row_id].values for k, f in files.items()},
                 index=sample.row_id)
R = P.rank(pct=True)
print('순위 상관 (Spearman):'); print(P.corr(method='spearman').round(3))

순위 상관 (Spearman):
       s15     s2     s3     s4     s5
s15  1.000  0.809  0.744  0.818  0.743
s2   0.809  1.000  0.901  0.966  0.859
s3   0.744  0.901  1.000  0.892  0.967
s4   0.818  0.966  0.892  1.000  0.906
s5   0.743  0.859  0.967  0.906  1.000


In [2]:
def save(score, fname):
    score = np.asarray(score, float) + 1e-6 * R.s15.values + 1e-9 * R.s4.values   # 동점 깨기
    s = pd.Series(rankdata(score) / len(score), index=P.index)
    s = (s - s.min()) / (s.max() - s.min())
    sub = pd.DataFrame({'row_id': P.index, 'prediction': s.values})
    assert list(sub.columns) == ['row_id', 'prediction'] and len(sub) == 694 and sub.row_id.is_unique
    assert set(sub.row_id) == set(sample.row_id) and sub.prediction.between(0, 1).all() and sub.prediction.notna().all()
    sub.to_csv(f'{SUB}/{fname}', index=False, encoding='utf-8')
    print(fname, '| 동점 행', int(sub.prediction.duplicated(keep=False).sum()))

save(0.5 * R.s4 + 0.5 * R.s15,              'submit_18_ens_s4_s15.csv')
save(0.5 * R.s3 + 0.5 * R.s15,              'submit_19_ens_s3_s15.csv')
save((R.s4 + R.s2 + R.s15) / 3,             'submit_20_ens_s4_s2_s15.csv')
save(0.35 * R.s4 + 0.35 * R.s3 + 0.3 * R.s15, 'submit_21_ens_s4_s3_s15.csv')

submit_18_ens_s4_s15.csv | 동점 행 0
submit_19_ens_s3_s15.csv | 동점 행 0
submit_20_ens_s4_s2_s15.csv | 동점 행 0
submit_21_ens_s4_s3_s15.csv | 동점 행 0
